In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
import os

import re 
from nltk.corpus import stopwords
from string import punctuation
from nltk.tokenize import word_tokenize

In [ ]:
df = pd.read_csv('arabic_sentiment_reviews.csv')

In [ ]:
df = df.sample(frac = 1 , random_state = 101)

In [ ]:
df.head(10)

In [ ]:
len(df)

In [ ]:
# positive -> 1 \ negative -> 0
np.unique(df['label'])

In [ ]:
id2label = {0 : "Negative", 1 : "Positive"}
label2id = {"Negative" : 0, "Positive" : 1}

In [ ]:
print(f"There are {df.duplicated().sum()} of duplicates in the dataset")
df.drop_duplicates(inplace=True)
print(f"There are {df.duplicated().sum()} of duplicates in the dataset")

In [ ]:
sns.set_theme(style='darkgrid', palette='pastel')
color = sns.color_palette(palette='pastel')
puncs = list(punctuation)
stop_words = set(stopwords.words('arabic'))

In [ ]:
def clean_text(text) : 
    if not isinstance(text , str) : 
        return ""

    # Remove URLs, Emails, and Mentions
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#', '', text)
    
    # digits with no space
    txt = re.sub(r"\d", ' ', text) 

    # Remove Punctuation and Numbers 
    text = re.sub(r'[^\s\u0600-\u06FF]', ' ', text)

    # set space before and after any punctuation
    text = re.sub(r"([^\w\s])", r" \1 ", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text)

    # 7. Remove Repeating Characters (e.g., "رااااائع" -> "رائع")
    text = re.sub(r'(.)\1+', r'\1', text)

    # Remove Arabic Diacritics (Tashkeel)
    tashkeel_pattern = re.compile(r'[\u0617-\u061A\u064B-\u0652]')
    text = tashkeel_pattern.sub('', text)

    # Normalization (Unifying alphabet forms)
    text = re.sub(r"[إأآا]", "ا", text) # Normalize Alefs
    text = re.sub(r"ة", "ه", text)      # Normalize Ta Marbuta
    text = re.sub(r"ى", "ي", text)      # Normalize Alef Maqsura to Ya

    # Remove Tatweel (Kashida) - e.g., "كــــــتاب" -> "كتاب"
    text = re.sub(r'ـ', '', text)

    # tokenization 
    tokens = word_tokenize(text)
    # remove punctuations
    tokens = [word for word in tokens if word not in puncs]
    # remove stopwords
    text = ' '.join([word for word in tokens if word not in stop_words])
    return text.lower().strip()

In [ ]:
df['cleaned_content'] = df['content'].apply(clean_text)

In [ ]:
print(f"Original content : {df['content'].iloc[0]}")
print(f"Cleaned content : {df['cleaned_content'].iloc[0]}")

In [ ]:
df['txt_len'] = [len(text.split()) for text in df.cleaned_content]

In [ ]:
f, (ax_box, ax_hist) = plt.subplots(2, sharex=True, 
                                    gridspec_kw={"height_ratios": (.15, .85)}, 
                                    figsize=(10, 7))
sns.boxplot(data=df, x="txt_len", ax=ax_box, color='lightblue')
ax_box.set(xlabel='') 
sns.histplot(data=df, x="txt_len", ax=ax_hist, kde=True)

f.suptitle("Reviews Length Distribution", y=0.92) 
plt.show()
df['txt_len'].describe()

In [ ]:
print(f"Number of records before : {len(df)}")
df = df[df['txt_len'].between(6 , 130)]
print(f"Number of records after : {len(df)}")

In [ ]:
label_counts = df['label'].value_counts()
chart_labels = [id2label[i] for i in label_counts.index]
wedges , _ , _  = plt.pie(label_counts, labels = chart_labels , autopct = '%1.1f%%') 
plt.legend(wedges , chart_labels , loc = 'right' , bbox_to_anchor = (1 , 0 , 0.5 , 1))
plt.show()

In [ ]:
df.head()

In [ ]:
df.to_json('filtered_arabic_sentiment_reviews.json')

In [ ]:
df.to_csv("filtered_arabic_sentiment_reviews.csv")